# Trust but Verify, Step 12: open-weights extraction on Kaggle

Runs **Qwen2.5-7B-Instruct-AWQ** (runbook default) with vLLM on Kaggle's free GPU and extracts every
synthetic and adversarial document with K = 5. No PCSS tokens are used.

**Before running**
1. Notebook settings → *Accelerator*: **GPU T4 x2**; *Internet*: **on** (needs a phone-verified account).
2. *Add data* → your dataset **tbv-data** (a zip of `data/synthetic` and `data/adversarial` from your laptop; see the last cell).
3. Optional, to resume after the 12-hour limit: add the output of the previous run as a dataset named **tbv-cache**.
4. If the GitHub repo is private: *Add-ons → Secrets* → `GITHUB_TOKEN` (a read-only token).

Only synthetic and adversarial (public, generated) documents go to Kaggle; never real ones.


In [ ]:
import os, subprocess, glob
REPO = "https://github.com/donti507/Trust-but-Verify.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = REPO.replace("https://", f"https://x-access-token:{token}@")
except Exception:
    pass  # public repo
!rm -rf /kaggle/working/tbv && git clone -q {REPO} /kaggle/working/tbv
%cd /kaggle/working/tbv
!git log --oneline -1

In [ ]:
!apt-get -qq update && apt-get -qq install -y libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz-subset0 > /dev/null
!pip -q install -r requirements-dev.txt && pip -q install -e . && pip -q install vllm
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Data (and previous cache, to resume)

In [ ]:
import shutil, pathlib
src = next(p for p in glob.glob('/kaggle/input/**/synthetic/docs.jsonl', recursive=True))
data_root = pathlib.Path(src).parents[1]
!rm -rf data && mkdir -p data && cp -r {data_root}/synthetic {data_root}/adversarial data/
prev = glob.glob('/kaggle/input/tbv-cache/**/cache', recursive=True)
if prev:
    !mkdir -p cache && cp -rn {prev[0]}/* cache/
    print("resumed from", prev[0])
!wc -l data/synthetic/docs.jsonl data/adversarial/docs.jsonl

## Start vLLM (OpenAI-compatible server on this machine)

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--quantization", "awq", "--dtype", "half", "--tensor-parallel-size", "2",
     "--max-model-len", "12288", "--gpu-memory-utilization", "0.92", "--port", "8000", "--api-key", "local"],
    stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT)
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5); print("vLLM ready"); break
    except Exception:
        time.sleep(10)
else:
    print(open("/kaggle/working/vllm.log").read()[-3000:])

## Pin the model (probes JSON mode and log-probs, writes configs/models.yaml)

In [ ]:
os.environ["OPEN_WEIGHTS_BASE_URL"] = "http://localhost:8000/v1"
os.environ["OPEN_WEIGHTS_API_KEY"] = "local"
from huggingface_hub import model_info
rev = model_info(MODEL).sha
!sed -i '/^  open_weights:/,$ s/^    extra_body: .*/    extra_body: {}/' configs/models.yaml
!python scripts/pin_model.py --track open_weights --name {MODEL} --version {MODEL}@{rev}

## Extract everything (K = 5), then verifier, injection and checks
Interrupted? Save the output, attach it as **tbv-cache**, and run again: cached calls are skipped.

In [ ]:
!python scripts/full_extraction.py run --model open_weights --k 5 --workers 16 --yes

In [ ]:
!python scripts/full_extraction.py verify --model open_weights
!python scripts/full_extraction.py inject --model open_weights
!python scripts/full_extraction.py check --model open_weights --k 5

## Save results (download from the Output tab)

In [ ]:
!zip -qr /kaggle/working/s12_open_weights.zip cache/raw/open_weights cache/extract/open_weights \
    cache/verify/open_weights cache/extract_injected/open_weights cache/verify_injected/open_weights \
    results configs/models.yaml
!ls -lh /kaggle/working/s12_open_weights.zip
server.terminate()

## Making the `tbv-data` dataset (run on your laptop, not here)
```bash
cd /mnt/e/projects/Trust-but-Verify
zip -qr tbv-data.zip data/synthetic data/adversarial -x "*.pdf" "*.html"
```
Then kaggle.com → *Datasets* → *New dataset* → upload `tbv-data.zip`, name it **tbv-data**, keep it **private**.